# 逻辑回归第三课：梯度与参数更新

这份 Notebook 专门讲逻辑回归中的梯度是怎么来的，以及参数如何通过梯度下降更新。

这一课的目标是把上一课的交叉熵真正接到训练过程上。

## 1. 先回顾上一课的核心公式

逻辑回归的预测过程可以写成：

$$
z = w^T x + b
$$

$$
p = \sigma(z) = \frac{1}{1+e^{-z}}
$$

其中 $p$ 表示正类概率：

$$
p = P(y=1 \mid x)
$$

单个样本的交叉熵损失是：

$$
L(y,p) = -[y\log p + (1-y)\log(1-p)]
$$

现在的问题是：

- 这个损失函数对参数 $w,b$ 的梯度是什么？
- 得到梯度之后，参数应该如何更新？

## 2. 为什么要求梯度

训练模型的目标，是让整体损失尽可能小。

如果我们知道损失函数对参数的梯度，就能知道：

- 参数往哪个方向改，损失会下降
- 每次改多少，由学习率决定

这就是梯度下降的核心思想。

因此这一课的关键是先求出：

$$
\frac{\partial L}{\partial w}, \qquad \frac{\partial L}{\partial b}
$$


## 3. 先求单个样本的梯度

先只看一个样本。因为一个样本的推导清楚了，推广到整个数据集只是把所有样本加总再取平均。

单个样本中：

$$
z = w^T x + b
$$

$$
p = \sigma(z)
$$

$$
L(y,p) = -[y\log p + (1-y)\log(1-p)]
$$

由于 $L$ 不是直接写成 $w$ 的函数，而是经过了

$$
w \rightarrow z \rightarrow p \rightarrow L
$$

所以这里要用链式法则。

## 4. 第一步：求 $\frac{\partial L}{\partial p}$

从

$$
L(y,p) = -[y\log p + (1-y)\log(1-p)]
$$

对 $p$ 求导：

$$
\frac{\partial L}{\partial p} = -\left(\frac{y}{p} - \frac{1-y}{1-p}\right)
$$

这一步只是普通求导。

- $\log p$ 的导数是 $\frac{1}{p}$
- $\log(1-p)$ 的导数是 $-\frac{1}{1-p}$

## 5. 第二步：求 $\frac{\partial p}{\partial z}$

由于

$$
p = \sigma(z) = \frac{1}{1+e^{-z}}
$$

sigmoid 的导数是一个非常重要的结论：

$$
\frac{\partial p}{\partial z} = p(1-p)
$$

这个公式后面会反复出现，所以最好直接记住。

## 6. 第三步：求 $\frac{\partial L}{\partial z}$

根据链式法则：

$$
\frac{\partial L}{\partial z} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial z}
$$

代入前两步的结果：

$$
\frac{\partial L}{\partial z} = -\left(\frac{y}{p} - \frac{1-y}{1-p}\right) p(1-p)
$$

展开整理：

$$
\frac{\partial L}{\partial z} = -(y(1-p) - (1-y)p)
$$

$$
\frac{\partial L}{\partial z} = -(y - yp - p + yp)
$$

$$
\frac{\partial L}{\partial z} = p - y
$$

这就是逻辑回归里最关键的化简结果。

它很重要，因为原本看起来很复杂的交叉熵和 sigmoid 组合，最后竟然会化简成非常简单的：

$$
\frac{\partial L}{\partial z} = p - y
$$


## 7. 第四步：求 $\frac{\partial L}{\partial w}$ 和 $\frac{\partial L}{\partial b}$

因为

$$
z = w^T x + b
$$

所以：

$$
\frac{\partial z}{\partial w} = x
$$

$$
\frac{\partial z}{\partial b} = 1
$$

继续使用链式法则：

$$
\frac{\partial L}{\partial w} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial w} = (p-y)x
$$

$$
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial b} = p-y
$$

因此，单个样本的梯度结果是：

$$
\nabla_w L = (p-y)x
$$

$$
\frac{\partial L}{\partial b} = p-y
$$


## 8. 推广到整个训练集

对于整个训练集，平均交叉熵损失写成：

$$
J(w,b) = \frac{1}{m}\sum_{i=1}^{m} L^{(i)}
$$

因此，把单个样本的梯度加总再取平均，就得到：

$$
\frac{\partial J}{\partial w} = \frac{1}{m}\sum_{i=1}^{m} (p^{(i)} - y^{(i)})x^{(i)}
$$

$$
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m} (p^{(i)} - y^{(i)})
$$

这就是标量写法。

## 9. 向量化写法

设：

$$
X \in \mathbb{R}^{m \times n}
$$

$$
w \in \mathbb{R}^{n}
$$

$$
y \in \mathbb{R}^{m}
$$

则：

$$
z = Xw + b
$$

$$
p = \sigma(z)
$$

这里的 $p$ 和 $y$ 都是长度为 $m$ 的向量。

于是梯度可以写成：

$$
\nabla_w J = \frac{1}{m}X^T(p-y)
$$

$$
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}(p^{(i)}-y^{(i)})
$$

这就是后面写 NumPy 代码时最常用的形式。

## 10. 参数如何更新

有了梯度之后，就可以使用梯度下降。

更新公式是：

$$
w := w - \alpha \nabla_w J
$$

$$
b := b - \alpha \frac{\partial J}{\partial b}
$$

代入上一节的结果：

$$
w := w - \alpha \cdot \frac{1}{m}X^T(p-y)
$$

$$
b := b - \alpha \cdot \frac{1}{m}\sum_{i=1}^{m}(p^{(i)}-y^{(i)})
$$

其中 $\alpha$ 是学习率。

- 太大，可能震荡甚至发散
- 太小，下降会很慢

## 11. 从结果上怎样理解 $p-y$

$p-y$ 这个量可以看成“预测概率和真实标签之间的误差”。

例如：

- 如果真实标签 $y=1$，而模型只给出 $p=0.2$，那么

$$
p-y = 0.2-1 = -0.8
$$

说明模型把正类概率估得太低了，参数需要朝增大正类概率的方向调整。

- 如果真实标签 $y=0$，而模型给出 $p=0.9$，那么

$$
p-y = 0.9-0 = 0.9
$$

说明模型把正类概率估得太高了，参数需要朝减小正类概率的方向调整。

所以这个结果既简洁，又非常符合直觉。

## 12. 本节小结

这一课最关键的推导链条是：

$$
L(y,p) = -[y\log p + (1-y)\log(1-p)]
$$

$$
\frac{\partial L}{\partial z} = p-y
$$

$$
\nabla_w J = \frac{1}{m}X^T(p-y)
$$

$$
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}(p^{(i)}-y^{(i)})
$$

$$
w := w - \alpha \nabla_w J
$$

$$
b := b - \alpha \frac{\partial J}{\partial b}
$$

到这里为止，你已经把逻辑回归的损失函数和训练更新真正接起来了。

下一步最自然的内容，就是用 NumPy 手写一个最小版逻辑回归。

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_loss(y, p):
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

X = np.array([
    [0.0, 1.0],
    [1.0, 1.0],
    [2.0, 1.0],
    [3.0, 1.0]
])

y = np.array([0.0, 0.0, 1.0, 1.0])

w = np.zeros(X.shape[1])
b = 0.0
alpha = 0.1

for epoch in range(1, 11):
    z = X @ w + b
    p = sigmoid(z)
    loss = compute_loss(y, p)

    dw = (X.T @ (p - y)) / len(y)
    db = np.mean(p - y)

    w -= alpha * dw
    b -= alpha * db

    print(f"epoch={epoch:2d}, loss={loss:.6f}, w={w}, b={b:.6f}")
